In [1]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/diabetic_data.csv")

print(df.shape)                # Number of rows and columns
print(df.columns.tolist())     # List column names
df.head()                      # First 5 rows

(101766, 50)
['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [4]:
df.info()     # Column types and non-null counts

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

In [5]:
df = pd.read_csv("../data/diabetic_data.csv", na_values="?", low_memory=False)
df.isna().sum()     # Show actual missing counts per column

encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum               96420
A1Cresult                   84748
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

In [6]:
df["readmitted"].value_counts()     # Number of readmits

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

In [7]:
# Percentage of data missing for missing-value columns 
missing_pct = df.isna().sum() / len(df) * 100
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64


In [8]:
# Drop weight
df = df.drop(columns=["weight"])

# Keep test results as a category
df["max_glu_serum"] = df["max_glu_serum"].fillna("Not tested")
df["A1Cresult"] = df["A1Cresult"].fillna("Not tested")

# Fill missing rows with "Unknown"
df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")
df["payer_code"] = df["payer_code"].fillna("Unknown")
df["race"] = df["race"].fillna("Unknown")

# Fill with "None" if no secondary/tertiary diagnosis
df["diag_3"] = df["diag_3"].fillna("None")
df["diag_2"] = df["diag_2"].fillna("None")
df["diag_1"] = df["diag_1"].fillna("None")

# Check that no missing values remain
df.isna().sum().sum()

np.int64(0)

In [9]:
# Target variable
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)
df["readmitted_30d"].value_counts()

readmitted_30d
0    90409
1    11357
Name: count, dtype: int64

In [10]:
# Pick features
feature_cols = ["race", "gender", "age", "number_outpatient", "number_emergency",
                "number_inpatient", "time_in_hospital", "num_lab_procedures",
                "num_medications", "number_diagnoses", "max_glu_serum",
                "A1Cresult", "diag_1", "discharge_disposition_id"]
df[feature_cols].head()

,race,gender,age,number_outpatient,number_emergency,number_inpatient,time_in_hospital,num_lab_procedures,num_medications,number_diagnoses,max_glu_serum,A1Cresult,diag_1,discharge_disposition_id
0,Caucasian,Female,[0-10),0,0,0,1,41,1,1,Not tested,Not tested,250.83,25
1,Caucasian,Female,[10-20),0,0,0,3,59,18,9,Not tested,Not tested,276,1
2,AfricanAmerican,Female,[20-30),2,0,1,2,11,13,6,Not tested,Not tested,648,1
3,Caucasian,Male,[30-40),0,0,0,2,44,16,7,Not tested,Not tested,8,1
4,Caucasian,Male,[40-50),0,0,0,1,51,8,5,Not tested,Not tested,197,1


In [11]:
# Ordinally encode age
age_order = ["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
             "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]
age_dict = {}
for i, label in enumerate(age_order):
    age_dict[label] = i
df["age_encoded"] = df["age"].map(age_dict)

# max_glu_serum and A1Cresult in order of severity 
glucose_order = {"Not tested": 0, "Norm": 1, ">200": 2, ">300": 3}
df["max_glu_serum_encoded"] = df["max_glu_serum"].map(glucose_order)

a1c_order = {"Not tested": 0, "Norm": 1, ">7": 2, ">8": 3}
df["A1Cresult_encoded"] = df["A1Cresult"].map(a1c_order)

In [12]:
# Ordering is not relevant
df = pd.get_dummies(df, columns=["race", "gender"], drop_first=True)

In [13]:
print(df["diag_1"].nunique())
print(df["discharge_disposition_id"].nunique())

717
26


In [14]:
print(df["discharge_disposition_id"].value_counts())

discharge_disposition_id
1     60234
3     13954
6     12902
18     3691
2      2128
22     1993
11     1642
5      1184
25      989
4       815
7       623
23      412
13      399
14      372
28      139
8       108
15       63
24       48
9        21
17       14
16       11
19        8
10        6
27        5
12        3
20        2
Name: count, dtype: int64


In [15]:
# Classify diagnosis using ICD-9 codes
def map_diagnosis(code):
    try:
        code = str(code)
        if code.startswith("250"):
            return "Diabetes"
        code_num = float(code)
        if 390 <= code_num <= 459 or code_num == 785:
            return "Circulatory"
        elif 460 <= code_num <= 519 or code_num == 786:
            return "Respiratory"
        elif 520 <= code_num <= 579 or code_num == 787:
            return "Digestive"
        elif 800 <= code_num <= 999:
            return "Injury"
        elif 710 <= code_num <= 739:
            return "Musculoskeletal"
        elif 580 <= code_num <= 629 or code_num == 788:
            return "Genitourinary"
        elif 140 <= code_num <= 239:
            return "Neoplasms"
        else:
            return "Other"
    except ValueError:
        return "Other"     # for codes that are non-numeric

df["diag_1_group"] = df["diag_1"].apply(map_diagnosis)
df["diag_1_group"].value_counts()

diag_1_group
Circulatory        30437
Other              18193
Respiratory        14423
Digestive           9475
Diabetes            8757
Injury              6974
Genitourinary       5117
Musculoskeletal     4957
Neoplasms           3433
Name: count, dtype: int64

In [16]:
# Drop rows with expired discharge disposition id
df = df[~df['discharge_disposition_id'].isin([11, 19, 20, 21])]
print(df.shape)

(100114, 59)


In [17]:
df = pd.get_dummies(df, columns=["diag_1_group", "discharge_disposition_id"], drop_first=True) # to avoid multicollinearity

In [18]:
# Get column names of encoded versions
print([c for c in df.columns if c.startswith("diag_1_group_")])
print([c for c in df.columns if c.startswith("discharge_disposition_id_")])
print([c for c in df.columns if c.startswith("race_")])
print([c for c in df.columns if c.startswith("gender_")])

['diag_1_group_Diabetes', 'diag_1_group_Digestive', 'diag_1_group_Genitourinary', 'diag_1_group_Injury', 'diag_1_group_Musculoskeletal', 'diag_1_group_Neoplasms', 'diag_1_group_Other', 'diag_1_group_Respiratory']
['discharge_disposition_id_2', 'discharge_disposition_id_3', 'discharge_disposition_id_4', 'discharge_disposition_id_5', 'discharge_disposition_id_6', 'discharge_disposition_id_7', 'discharge_disposition_id_8', 'discharge_disposition_id_9', 'discharge_disposition_id_10', 'discharge_disposition_id_12', 'discharge_disposition_id_13', 'discharge_disposition_id_14', 'discharge_disposition_id_15', 'discharge_disposition_id_16', 'discharge_disposition_id_17', 'discharge_disposition_id_18', 'discharge_disposition_id_22', 'discharge_disposition_id_23', 'discharge_disposition_id_24', 'discharge_disposition_id_25', 'discharge_disposition_id_27', 'discharge_disposition_id_28']
['race_Asian', 'race_Caucasian', 'race_Hispanic', 'race_Other', 'race_Unknown']
['gender_Male', 'gender_Unknown/

In [19]:
# Gender dropped female, race dropped AfricanAmerican

In [20]:
feature_cols = (
    ["age_encoded", "max_glu_serum_encoded", "A1Cresult_encoded",
     "number_outpatient", "number_emergency", "number_inpatient",
     "time_in_hospital", "num_lab_procedures", "num_medications", "number_diagnoses"]
    + ["diag_1_group_Diabetes", "diag_1_group_Digestive", "diag_1_group_Genitourinary",
       "diag_1_group_Injury", "diag_1_group_Musculoskeletal", "diag_1_group_Neoplasms",
       "diag_1_group_Other", "diag_1_group_Respiratory"]
    + ["discharge_disposition_id_2", "discharge_disposition_id_3", "discharge_disposition_id_4",
       "discharge_disposition_id_5", "discharge_disposition_id_6", "discharge_disposition_id_7",
       "discharge_disposition_id_8", "discharge_disposition_id_9", "discharge_disposition_id_10",
       "discharge_disposition_id_12", "discharge_disposition_id_13", "discharge_disposition_id_14",
       "discharge_disposition_id_15", "discharge_disposition_id_16", "discharge_disposition_id_17",
       "discharge_disposition_id_18", "discharge_disposition_id_22", "discharge_disposition_id_23",
       "discharge_disposition_id_24", "discharge_disposition_id_25", "discharge_disposition_id_27",
       "discharge_disposition_id_28"]
    + ["race_Asian", "race_Caucasian", "race_Hispanic", "race_Other", "race_Unknown"]
    + ["gender_Male", "gender_Unknown/Invalid"]
)

print(len(feature_cols))

47


In [21]:
# Check feature_cols for errors
missing = [c for c in feature_cols if c not in df.columns]
print(missing)

[]
